# Draft a Unified Table for **Points of Interest** in Berlin Layers

## Step 1: Gather info on tables already in the database

### Import libraries

In [1]:
# Import Libraries
import osmnx as ox # to fetch data from OpenStreetMap
import geopandas as gpd # to work with geospatial data
import pandas as pd
# import psycopg2
from sqlalchemy import create_engine, text
import warnings
import json

warnings.filterwarnings("ignore")

### Credentials

In [ ]:
user_name=''
password='f'

### Create the connection

In [3]:
# Conection
host = 'localhost'
port = '5433'
database = 'layereddb'
schema='berlin_source_data'

#connection to db after you opened tunnel
engine = create_engine(f'postgresql+psycopg2://{user_name}:{password}@{host}:{port}/{database}')

### Queries

- Show tables list that are not statitics

In [4]:
# Get all table names
query = """
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'berlin_source_data'
  AND table_type = 'BASE TABLE'
  AND table_name NOT ILIKE '%stat%'
"""
# Execute the query
poi_tables_df = pd.read_sql(text(query), engine)
print(poi_tables_df)

                          table_name
0                          galleries
1                            museums
2                           theaters
3                   pools_refactored
4                    public_artworks
5        theaters_backup_neigh_final
6          district_level_aggregated
7                              malls
8                     bus_tram_stops
9                            doctors
10                             banks
11           social_clubs_activities
12  veterinary_clinics_martin_svitek
13              hospitals_refactored
14                        pharmacies
15                        bike_lanes
16                      supermarkets
17                             pools
18                       land_prices
19          test_table_george_smelin
20                            venues
21                      post_offices
22                    dental_offices
23                         districts
24                              gyms
25               short_term_listings
2

- Show tables list that are only statitics

In [5]:
query = f"""
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'berlin_source_data'
    AND table_name LIKE '%stat%'
ORDER BY table_name;
"""

# Execute the query
stats_tables_df = pd.read_sql(text(query), engine)
print(stats_tables_df)

                    table_name
0             crime_statistics
1           districts_pop_stat
2          regional_statistics
3  rent_stats_per_neighborhood


In [6]:
# Loop through tables and show top 5 rows of stats tables
for table in stats_tables_df['table_name']:
    print(f"\n--- {table} ---")
    preview_query = f"SELECT * FROM berlin_source_data.{table} LIMIT 5;"
    df = pd.read_sql(text(preview_query), engine)
    display(df)


--- crime_statistics ---


,id,area_id,locality,district,district_id,year,crime_type_german,crime_type_english,category,total_number_cases,frequency_100k,population_base,severity_weight,created_at,updated_at
0,2001,13007.0,Osloer Straße,Mitte,11001001,2015,Brand- stiftung,Arson,Property Crime,13,34.0,None,4.5,2025-08-21 12:52:44.976746,2025-08-21 12:52:44.976746
1,2002,13008.0,Brunnenstraße Nord,Mitte,11001001,2015,Brand- stiftung,Arson,Property Crime,10,27.0,None,4.5,2025-08-21 12:52:44.976746,2025-08-21 12:52:44.976746
2,2003,14009.0,Parkviertel,Mitte,11001001,2015,Brand- stiftung,Arson,Property Crime,16,36.0,None,4.5,2025-08-21 12:52:44.976746,2025-08-21 12:52:44.976746
3,2004,14010.0,Wedding Zentrum,Mitte,11001001,2015,Brand- stiftung,Arson,Property Crime,14,25.0,None,4.5,2025-08-21 12:52:44.976746,2025-08-21 12:52:44.976746
4,2005,19900.0,"Bezirk (Mi), nicht zuzuordnen",Mitte,11001001,2015,Brand- stiftung,Arson,Property Crime,8,NaN,None,4.5,2025-08-21 12:52:44.976746,2025-08-21 12:52:44.976746



--- districts_pop_stat ---


,district_id,district,male,female,germans,foreigners,single,married,widowed,divorced,...,römisch_katholische_kirche,religion_other_or_none,0-6,6-15,15-18,18-27,27-45,45-55,55-65,65+
0,11001001,Mitte,204542,192462,249053,147951,240196,113277,13119,28623,...,28294,333248,20516,29120,9461,49247,145300,47278,44361,51721
1,11002002,Friedrichshain-Kreuzberg,150002,142622,202632,89992,188392,76076,7369,19227,...,17094,251355,14918,21510,6836,30281,114196,40160,32574,32149
2,11003003,Pankow,209773,217503,343105,84171,246687,130125,17014,31848,...,23544,363021,22768,38101,12347,39397,132140,62325,56884,63314
3,11004004,Charlottenburg-Wilmersdorf,166037,177463,249994,93506,176967,114706,17823,32281,...,31889,263824,15921,23364,7968,35224,94225,39648,47924,79226
4,11005005,Spandau,127361,131916,189914,69363,125965,93781,15779,23333,...,21050,198091,15354,24680,7873,26455,66010,29564,35687,53654



--- regional_statistics ---


,district_id,district,year,inhabitants,total_area_ha,share_forest_water_agriculture,forest_area_ha,water_area_ha,agriculture_area_ha,population_density_per_ha,number_of_residences,living_space_per_resident_m2
0,11004004,Charlottenburg-Wilmersdorf,2012,298567,6472,0.295,1622,281,794,46.1,184637,48.9
1,11002002,Friedrichshain-Kreuzberg,2012,259483,2034,0.067,4,132,1,127.6,145328,39.3
2,11011011,Lichtenberg,2012,258586,5212,0.140,51,104,293,49.6,143895,37.0
3,11010010,Marzahn-Hellersdorf,2012,248786,6178,0.060,173,117,573,40.3,127149,37.1
4,11001001,Mitte,2012,329969,3947,0.036,0,142,0,83.6,185209,39.2



--- rent_stats_per_neighborhood ---


,district_id,district,median_net_rent_per_m2,number_of_cases,mean_net_rent_per_m2,year
0,11001001,Mitte,19.91,4169,19.91,2024
1,11002002,Friedrichshain-Kreuzberg,19.42,2509,19.26,2024
2,11003003,Pankow,17.00,3636,17.65,2024
3,11004004,Charlottenburg-Wilmersdorf,19.39,3036,19.40,2024
4,11005005,Spandau,12.00,2403,13.32,2024


In [7]:
# Loop through tables and show top 5 rows of point of interest tables
for table in poi_tables_df['table_name']:
    print(f"\n--- {table} ---")
    preview_query = f"SELECT * FROM berlin_source_data.{table} LIMIT 5;"
    df = pd.read_sql(text(preview_query), engine)
    display(df)


--- galleries ---


,id,name,house_number,street,neighborhood_id,district_id,postal_code,website,opening_hours,wheelchair,fee,latitude,longitude
0,301107444,atelier achim kühn,6,richter straße,0908,11009009,12524,None,None,None,None,52.409750,13.571342
1,370766098,galerie zandi,57,mommsen straße,0401,11004004,10629,None,mo-fr 11:00-18:00; sa 11:00-16:00,no,false,52.503135,13.313631
2,410135935,studio für gestaltung,31,bürkner straße,0801,11008008,12047,None,"th,fr 13:00-18:00; sa 12:00-16:00",yes,false,52.493327,13.427490
3,410692505,la girafe,None,None,0202,11002002,None,None,None,no,false,52.492914,13.437057
4,410745800,kronenboden,16,schweden straße,0106,11001001,13357,None,None,yes,None,52.555091,13.375326



--- museums ---


,id,name,house_number,street,neighborhood_id,district_id,postal_code,website,normalized_phone,museum_type,operator,building,wikipedia,opening_hours,wheelchair,toilets_wheelchair_accessible,fee,latitude,longitude
0,73696610,gedenkstätte deutscher widerstand,13-14,stauffenberg straße,0104,11001001,10785,https://www.gdw-berlin.de/,None,None,None,None,de:gedenkstätte deutscher widerstand,mo-fr 09:00-18:00; sa-su 10:00-18:00,limited,None,false,52.507605,13.362986
1,84644782,"dokumentationszentrum flucht, vertreibung, ver...",90,stresemann straße,0202,11002002,10963,https://www.flucht-vertreibung-versoehnung.de/,None,None,None,None,"de:stiftung flucht, vertreibung, versöhnung",tu-su 10:00-19:00; mo off,yes,yes,false,52.504871,13.382214
2,259855486,berliner s-bahn-museum,3,koppen straße,0201,11002002,10243,https://s-bahn-museum.de/,None,railway,berliner s-bahn-museum ggmbh,None,None,"we 12:00-16:00, th,fr 15:00-20:00, su 14:00-18:00",yes,None,true,52.510475,13.432765
3,268591806,das museum der domäne dahlem,None,None,0605,11006006,None,https://www.domaene-dahlem.de/home/,None,open_air,None,None,None,mo-su 10:00-18:00; th off,limited,None,true,52.458357,13.288944
4,281391655,stasimuseum,103,rusche straße,1103,11011011,10365,https://www.stasimuseum.de/,+49 30 5536854,history,astak e.v.,None,de:forschungs- und gedenkstätte normannenstraße,mo-fr 10:00-18:00; sa-su 11:00-18:00,yes,None,true,52.514507,13.487485



--- theaters ---


...

,station,line,latitude,longitude,postcode,neighborhood,district,district_id
0,Adenauerplatz,U7,52.499722,13.307222,10707,Charlottenburg,Charlottenburg-Wilmersdorf,11004004
1,Afrikanische Straße,U6,52.560028,13.334633,13349,Wedding,Mitte,11001001
2,Alexanderplatz,U5,52.521389,13.411944,10178,Mitte,Mitte,11001001
3,Alexanderplatz,U2,52.521389,13.411944,10178,Mitte,Mitte,11001001
4,Alexanderplatz,U8,52.521389,13.411944,10178,Mitte,Mitte,11001001



--- playgrounds ---


,playground_id,name,latitude,longitude,district,district_id,neighborhood,neighborhood_id,area_sq_m,full_address
0,4675776,Unknown,52.533085,13.346343,Mitte,11001001,Moabit,0102,4247.857798,"Stephanstraße, Stephankiez, Moabit, Mitte, Ber..."
1,4675787,Unknown,52.534153,13.344207,Mitte,11001001,Moabit,0102,1488.792935,"Salzwedeler Straße, Stephankiez, Moabit, Mitte..."
2,4687127,Unknown,52.525066,13.473750,Lichtenberg,11011011,Fennpfuhl,1111,222.216294,"Paul-Junius-Straße, Fennpfuhl, Lichtenberg, Be..."
3,4703528,Unknown,52.533222,13.337152,Mitte,11001001,Moabit,0102,214.382142,"Unionstraße, Stephankiez, Moabit, Mitte, Berli..."
4,4707958,Spielplatz Zwinglistraße,52.525870,13.329238,Mitte,11001001,Moabit,0102,685.914246,"Spielplatz Zwinglistraße, 18, Zwinglistraße, B..."



--- exhibition_centers ---


,id,exhibition_center_name,house_number,street,neighborhood_id,district_id,postal_code,website,building,wikipedia,latitude,longitude
0,136079469,messe berlin,None,alemannen allee,0405,11004004,14052,https://www.messe-berlin.de/,None,de:messegelände (berlin),52.503449,13.272199
1,196694606,citycube berlin,26,messe damm,0405,11004004,14055,http://www.citycube-berlin.de/,yes,de:citycube berlin,52.500120,13.270794
2,680746621,hub27,None,jaffé straße,0405,11004004,14055,https://www.messe-berlin.de/de/veranstalter/un...,yes,None,52.502896,13.267529
3,719414532,ausstellungspavillon,None,None,1001,11010010,None,None,yes,None,52.539210,13.570826


### Check tables for certain columns

In [8]:
# Get columns with 'id', 'name', 'lat', 'lon' in their names
query = """
  WITH tables AS (
      SELECT table_name
      FROM information_schema.tables
      WHERE table_schema = 'berlin_source_data'
        AND table_type = 'BASE TABLE'
        AND table_name NOT ILIKE '%test%' AND table_name NOT ILIKE '%stat%'
  ),
  cols AS (
      SELECT table_name, column_name
      FROM information_schema.columns
      WHERE table_schema = 'berlin_source_data'
  )
  SELECT t.table_name,
        -- check for each required pattern
        bool_or(c.column_name ILIKE '%id%' AND c.column_name != 'district_id') AS has_id,
        bool_or(c.column_name ILIKE '%name%' OR c.column_name ILIKE '%station%' OR c.column_name ILIKE '%district%' ) AS has_name,
        bool_or(c.column_name ='district_id') AS has_district_id,
        bool_or(c.column_name ILIKE '%hood_id%') AS has_neighborhood_id,
        bool_or(c.column_name ILIKE '%lat%')  AS has_lat,
        bool_or(c.column_name ILIKE '%lon%')  AS has_lon,
        bool_or(c.column_name ILIKE '%geom%')  AS has_geom
  FROM tables t
  LEFT JOIN cols c ON t.table_name = c.table_name
  GROUP BY t.table_name
  ORDER BY t.table_name;
"""
# Execute the query
with engine.connect() as conn:
    df= pd.read_sql(text(query), conn)
    conn.commit()  # commit the transaction
df

,table_name,has_id,has_name,has_district_id,has_neighborhood_id,has_lat,has_lon,has_geom
0,banks,True,True,True,False,True,True,False
1,bike_lanes,True,True,True,True,False,False,True
2,bus_tram_stops,True,True,True,False,True,True,False
3,dental_offices,True,True,True,False,True,True,False
4,district_level_aggregated,True,True,True,False,True,True,False
5,districts,False,True,True,False,False,False,True
6,doctors,True,True,True,True,True,True,False
7,exhibition_centers,True,True,True,True,True,True,False
8,galleries,True,True,True,True,True,True,False
9,gyms,True,True,True,False,True,True,True


### Create a standard table for POI

In [84]:
from sqlalchemy import text

query = """

DROP TABLE IF EXISTS point_of_interest CASCADE;

CREATE TABLE point_of_interest (
    poi_id TEXT PRIMARY KEY,
    name TEXT,
    layer TEXT, -- 'galleries' or 'museums'
    district_id TEXT,
    district TEXT,
    neighborhood_id TEXT,
    neighborhood TEXT,
    latitude DOUBLE PRECISION,
    longitude DOUBLE PRECISION,
    geometry geometry(Point, 4326),
    attributes JSONB
);
"""

# Execute the DDL
with engine.begin() as conn:   # begin() handles commit automatically
    conn.execute(text(query))


### Insert rows into the POI - needs to be automated and poi_id to use the table name and not the actual word inserted, same for the layer column

In [85]:
insert_query = """
INSERT INTO point_of_interest (poi_id, name, layer, district_id, district, neighborhood_id, neighborhood,
                 latitude, longitude, geometry, attributes)
SELECT
    CONCAT(SUBSTRING('galleries' FROM 1 FOR 3), '-', g.id) AS poi_id,
    g.name,
    'galleries' AS layer,
    g.district_id,
    d.district AS district,
    g.neighborhood_id,
    n.neighborhood AS neighborhood,
    g.latitude,
    g.longitude,
    ST_SetSRID(ST_MakePoint(g.longitude, g.latitude), 4326) AS geometry,
    to_jsonb(g) - 'id' - 'name' - 'district_id' - 'neighborhood_id'
                 - 'latitude' - 'longitude' - 'geometry' AS attributes
FROM berlin_source_data.galleries g
JOIN berlin_source_data.districts d ON g.district_id = d.district_id
JOIN berlin_source_data.neighborhoods n ON g.neighborhood_id = n.neighborhood_id

UNION ALL

SELECT
    CONCAT(SUBSTRING('museums' FROM 1 FOR 3), '-', m.id) AS poi_id,
    m.name,
    'museums' AS layer,
    m.district_id,
    d.district AS district,
    m.neighborhood_id,
    n.neighborhood AS neighborhood,
    m.latitude,
    m.longitude,
    ST_SetSRID(ST_MakePoint(m.longitude, m.latitude), 4326) AS geometry,
    to_jsonb(m) - 'id' - 'name' - 'district_id' - 'neighborhood_id'
                 - 'latitude' - 'longitude' - 'geometry' AS attributes
FROM berlin_source_data.museums m
JOIN berlin_source_data.districts d ON m.district_id = d.district_id
JOIN berlin_source_data.neighborhoods n ON m.neighborhood_id = n.neighborhood_id

UNION ALL

SELECT
    CONCAT(SUBSTRING('listings' FROM 1 FOR 3), '-', l.listing_id) AS poi_id,
    l.detail_url as name,
    'listings' AS layer,
    l.district_id,
    d.district AS district,
    n.neighborhood_id,
    l.neighborhood AS neighborhood,
    l.latitude,
    l.longitude,
    ST_SetSRID(ST_MakePoint(l.longitude, l.latitude), 4326) AS geometry,
    to_jsonb(l) - 'id' - 'detail_url' - 'district_id' - 'neighborhood'
                 - 'latitude' - 'longitude' - 'geometry' AS attributes
FROM berlin_source_data.long_term_listings l
JOIN berlin_source_data.districts d ON l.district_id = d.district_id
JOIN berlin_source_data.neighborhoods n ON l.neighborhood = n.neighborhood

UNION ALL

SELECT
    CONCAT(SUBSTRING('banks' FROM 1 FOR 3), '-', b.bank_id) AS poi_id,
    b.name as name,
    'banks' AS layer,
    b.district_id,
    d.district AS district,
    '0000' AS neighborhood_id,
    'Unknown' AS  neighborhood,
    b.latitude,
    b.longitude,
    ST_SetSRID(ST_MakePoint(b.longitude, b.latitude), 4326) AS geometry,
    to_jsonb(b) - 'id' - 'detail_url' - 'district_id' - 'neighborhood'
                 - 'latitude' - 'longitude' - 'geometry' AS attributes
FROM berlin_source_data.banks b
JOIN berlin_source_data.districts d ON b.district_id = d.district_id
;
"""

with engine.begin() as conn:
    conn.execute(text(insert_query))

### Show 20 rows of POI table

In [66]:
query = f"""
SELECT poi_id,
    name,
    layer,
    district_id,
    district,
    neighborhood_id,
    neighborhood,
    latitude,
    longitude,
     ST_AsText(geometry) AS geom_wkt,
    attributes
FROM point_of_interest 
WHERE district = 'Spandau'
LIMIT 20;
"""

# Execute the query
with engine.connect() as conn:
    df= pd.read_sql(text(query), conn)
    conn.commit()  # commit the transaction
df

,poi_id,name,layer,district_id,district,neighborhood_id,neighborhood,latitude,longitude,geom_wkt,attributes
0,gal-10111146177,galerie im alten zollhaus,galleries,11005005,Spandau,0504,Staaken,52.522984,13.147897,POINT(13.147897 52.522984),"{'fee': None, 'street': 'heer straße', 'websit..."
1,gal-12734386032,kommunale galerie im historischen keller,galleries,11005005,Spandau,0501,Spandau,52.538544,13.204411,POINT(13.204411 52.538544),"{'fee': 'false', 'street': 'reformations platz..."
2,gal-26121688,kunstremise spandau,galleries,11005005,Spandau,0501,Spandau,52.537120,13.201191,POINT(13.201191 52.53712),"{'fee': None, 'street': 'viktoria ufer', 'webs..."
3,mus-344381412,enthüllt. berlin und seine denkmäler,museums,11005005,Spandau,0502,Haselhorst,52.541218,13.213854,POINT(13.213854 52.541218),"{'fee': 'true', 'street': 'am juliusturm', 'we..."
4,mus-2335146075,das gotische haus,museums,11005005,Spandau,0501,Spandau,52.537686,13.206724,POINT(13.206724 52.537686),"{'fee': None, 'street': None, 'website': 'http..."
5,mus-2853421751,spandovia sacra – museum mit bibliothek und ar...,museums,11005005,Spandau,0501,Spandau,52.538780,13.205653,POINT(13.205653 52.53878),"{'fee': None, 'street': None, 'website': None,..."
6,mus-3977880885,gutshof gatow,museums,11005005,Spandau,0505,Gatow,52.484940,13.181724,POINT(13.181724 52.48494),"{'fee': None, 'street': 'buchwald ziele', 'web..."
7,mus-8428338215,militär-historisches museum berlin-gatow,museums,11005005,Spandau,0506,Kladow,52.471926,13.138677,POINT(13.138677 52.471926),"{'fee': None, 'street': 'am flugplatz gatow', ..."
8,mus-25775881,stadtgeschichtliches museum spandau,museums,11005005,Spandau,0502,Haselhorst,52.540634,13.213536,POINT(13.213536 52.540634),"{'fee': 'true', 'street': None, 'website': Non..."
9,lis-WOH_759_42_13599,https://www.immowelt.de/expose/29290c8b-1ceb-4...,listings,11005005,Spandau,0502,Haselhorst,52.555490,13.224585,POINT(13.2245846 52.5554901),"{'city': 'Berlin', 'type': 'Wohnung', 'floor':..."


### Show some attributes of row 5 to see what it looks like

In [56]:
query = f"""
SELECT attributes
FROM point_of_interest 

LIMIT 20;
"""

# Execute the query
with engine.connect() as conn:
    df= pd.read_sql(text(query), conn)
    conn.commit()  # commit the transaction
df
print(json.dumps(df.loc[5, "attributes"], indent=2))

{
  "fee": "false",
  "street": "schierker stra\u00dfe",
  "website": null,
  "wheelchair": "yes",
  "postal_code": "12051",
  "house_number": "35",
  "opening_hours": "tu-th 11:00-22:00; fr,sa 11:00-23:00"
}


### ⚡ Indexing for Speed - GiST index on the geometry column

In [86]:
query = """
CREATE INDEX idx_poi_geom
ON point_of_interest
USING GIST (geometry);
"""

with engine.connect() as conn:
    conn.execute(text(query))   # run the CREATE INDEX
    conn.commit()               # commit the transaction


In [58]:
check_query = """
SELECT indexname, indexdef
FROM pg_indexes
WHERE tablename = 'point_of_interest';
"""
with engine.connect() as conn:
    testdf= pd.read_sql(text(check_query), conn)
    conn.commit()  # commit the transaction

testdf

,indexname,indexdef
0,point_of_interest_pkey,CREATE UNIQUE INDEX point_of_interest_pkey ON ...
1,idx_poi_geom,CREATE INDEX idx_poi_geom ON public.point_of_i...


### Look at long_term_listings table and hospitals bank table

In [59]:
query = """
SELECT *
FROM berlin_source_data.long_term_listings
LIMIT 2;
"""
# Execute the query
df = pd.read_sql(text(query), engine)
df

,listing_id,detail_url,raw_info,type,first_tenant,price_euro,number_of_rooms,surface_m2,floor,street,house_number,neighborhood,district,postal_code,city,address,latitude,longitude,geometry,district_id
0,WOH_2772_178_14055,https://www.immowelt.de/expose/fb32adca-5d67-4...,Wohnung zur Miete 2.772 € 4 Zimmer 178 m² frei...,Wohnung,no,2772,4.0,178.0,None,Stallupöner Allee,33,Westend,Charlottenburg-Wilmersdorf,14055,Berlin,Stallupöner Allee 33 14055 Westend Berlin,52.504417,13.234717,POINT (13.234717 52.504417),11004004
1,STU_495_43_13627,https://www.immowelt.de/expose/4985f014-94b9-4...,"Studio zur Miete 495 € 2 Zimmer 43,7 m² frei a...",Studio,no,495,2.0,43.7,None,Schweiggerweg,5,Charlottenburg-Nord,Charlottenburg-Wilmersdorf,13627,Berlin,Schweiggerweg 5 13627 Charlottenburg Berlin,52.538645,13.282277,POINT (13.2822767 52.5386453),11004004


In [60]:
query = """
SELECT *
FROM berlin_source_data.banks
LIMIT 2;
"""
# Execute the query
df = pd.read_sql(text(query), engine)
df

,bank_id,name,brand,operator,street,housenumber,postcode,opening_hours,atm,wheelchair,latitude,longitude,district,district_id
0,28968292,berliner volksbank,berliner volksbank,nan,berliner straße,42,10713,"mo-fr 10:00-13:00, mo 14:00-16:00, tu,th 14:00...",true,true,52.486668,13.319723,Charlottenburg-Wilmersdorf,11004004
1,60848455,sparkasse,nan,berliner sparkasse,anton-saefkow-platz,13,10369,"mo,we,fr 09:30-15:00; tu,th 09:30-18:00",true,unknown,52.530331,13.471037,Lichtenberg,11011011


### Show closest hospital to each listing

In [61]:
query = """
    SELECT l.listing_id,
        h.hospital_id AS hospital_id,
        h.name AS hospital_name,
        ST_DistanceSphere(ST_GeomFromText(l.geometry, 4326), ST_GeomFromText(h.geometry, 4326)) AS distance_meters
    FROM berlin_source_data.long_term_listings l
    JOIN LATERAL (
        SELECT hospital_id, name, geometry
        FROM berlin_source_data.hospitals_refactored h
        ORDER BY ST_GeomFromText(l.geometry, 4326) <-> ST_GeomFromText(h.geometry, 4326)   -- efficient nearest-neighbor using GiST index
        LIMIT 1
    ) h ON true;
"""
# Execute the query
df = pd.read_sql(text(query), engine)
df

,listing_id,hospital_id,hospital_name,distance_meters
0,WOH_2772_178_14055,138215763,Paulinen Krankenhaus,385.444188
1,STU_495_43_13627,114201147,Schlosspark-Klinik,1882.795097
2,STU_1625_44_10717,23484458,Ärztehaus am Hohenzollerndamm,234.123495
3,WOH_2400_132_14050,12218907397,Fertility Center Berlin,466.746290
4,WOH_1200_45_10589,114201147,Schlosspark-Klinik,745.140126
...,...,...,...,...
1162,HAU_2350_169_12587,5037228530,Ärzte Zentrum am Markt,931.006981
1163,STU_2500_169_12587,5037228530,Ärzte Zentrum am Markt,931.006981
1164,WOH_2398_109_12527,69869582,Krankenhaus Hedwigshöhe,4646.899357
1165,WOH_1840_76_14059,9178079418,Ärztehaus,644.476418


In [ ]:
query = """

"""
# Execute the query
df = pd.read_sql(text(query), engine)
df

### Create the json column to show nearest neighbor - can be automated so dont need to create a layer section each time

In [88]:
query = """
    ALTER TABLE point_of_interest  ADD COLUMN nearest_pois jsonb;

    UPDATE point_of_interest l
    SET nearest_pois = jsonb_build_object(
        'gallery', (
            SELECT jsonb_build_object(
                'id', h.poi_id, 
                'name', h.name, 
                'distance', ST_DistanceSphere(l.geometry, h.geometry),
                'address', jsonb_build_object('street', h.attributes->>'street', 'house_number', h.attributes->>'house_number'))
            FROM point_of_interest h
            WHERE h.layer = 'galleries'
            ORDER BY l.geometry <-> h.geometry
            LIMIT 1
        ),
        'museum', (
            SELECT jsonb_build_object(
                'id', m.poi_id, 
                'name', m.name, 
                'distance', ST_DistanceSphere(l.geometry, m.geometry),
                'address', jsonb_build_object('street', m.attributes->>'street', 'house_number', m.attributes->>'house_number'))
            FROM point_of_interest m
            WHERE m.layer = 'museums'
            ORDER BY l.geometry <-> m.geometry
            LIMIT 1
        ),
        'bank', (
            SELECT jsonb_build_object(
                'id', b.poi_id, 
                'name', b.name, 
                'distance', ST_DistanceSphere(l.geometry, b.geometry),
                'address', jsonb_build_object('street', b.attributes->>'street', 'house_number', b.attributes->>'house_number'))
            FROM point_of_interest b
            WHERE b.layer = 'banks'
            ORDER BY l.geometry <-> b.geometry
            LIMIT 1
        )
    )
    WHERE l.layer = 'listings';
"""
# Execute the query
with engine.begin() as conn:
    conn.execute(text(query))

### Run query to show 10 listings

In [89]:
query = f"""
SELECT *
FROM point_of_interest 
WHERE layer = 'listings'
LIMIT 10;
"""

# Execute the query
with engine.connect() as conn:
    df= pd.read_sql(text(query), conn)
    conn.commit()  # commit the transaction
df


,poi_id,name,layer,district_id,district,neighborhood_id,neighborhood,latitude,longitude,geometry,attributes,nearest_pois
0,lis-WOH_2772_178_14055,https://www.immowelt.de/expose/fb32adca-5d67-4...,listings,11004004,Charlottenburg-Wilmersdorf,0405,Westend,52.504417,13.234717,0101000020E610000066A19DD32C782A40F1457BBC9040...,"{'city': 'Berlin', 'type': 'Wohnung', 'floor':...","{'bank': {'id': 'ban-671776291', 'name': 'bank..."
1,lis-STU_495_43_13627,https://www.immowelt.de/expose/4985f014-94b9-4...,listings,11004004,Charlottenburg-Wilmersdorf,0406,Charlottenburg-Nord,52.538645,13.282277,0101000020E61000007AD8559286902A4072D24554F244...,"{'city': 'Berlin', 'type': 'Studio', 'floor': ...","{'bank': {'id': 'ban-437514581', 'name': 'spar..."
2,lis-STU_2716_80_14057,https://www.immowelt.de/expose/6a4f3322-2f3f-4...,listings,11004004,Charlottenburg-Wilmersdorf,0401,Charlottenburg,52.506960,13.286736,0101000020E6100000F52B9D0FCF922A4056325B0DE440...,"{'city': 'Berlin', 'type': 'Studio', 'floor': ...","{'bank': {'id': 'ban-1819283270', 'name': 'pos..."
3,lis-STU_1625_44_10717,https://www.immowelt.de/expose/2144d17c-ae55-4...,listings,11004004,Charlottenburg-Wilmersdorf,0402,Wilmersdorf,52.492051,13.320604,0101000020E610000052ED783826A42A4026BA7889FB3E...,"{'city': 'Berlin', 'type': 'Studio', 'floor': ...","{'bank': {'id': 'ban-2445456642', 'name': 'spa..."
4,lis-WOH_2400_132_14050,https://www.immowelt.de/expose/8d048a9f-4bf4-4...,listings,11004004,Charlottenburg-Wilmersdorf,0405,Westend,52.520175,13.269123,0101000020E61000004537B176CA892A4049DBF8139542...,"{'city': 'Berlin', 'type': 'Wohnung', 'floor':...","{'bank': {'id': 'ban-667211363', 'name': 'berl..."
5,lis-WOH_1200_45_10589,https://www.immowelt.de/expose/25ne5mmgni7e?ln...,listings,11004004,Charlottenburg-Wilmersdorf,0401,Charlottenburg,52.524411,13.300144,0101000020E61000006C335F6FAC992A402553AAE91F43...,"{'city': 'Berlin', 'type': 'Wohnung', 'floor':...","{'bank': {'id': 'ban-1915389761', 'name': 'spa..."
6,lis-WOH_1584_64_10719,https://www.immowelt.de/expose/1f6dafb1-fa25-4...,listings,11004004,Charlottenburg-Wilmersdorf,0402,Wilmersdorf,52.487919,13.323358,0101000020E610000022D04F278FA52A4000CC1022743E...,"{'city': 'Berlin', 'type': 'Wohnung', 'floor':...","{'bank': {'id': 'ban-2307254000', 'name': 'deu..."
7,lis-WOH_1850_83_10585,https://www.immowelt.de/expose/1a8a9d5a-9727-4...,listings,11004004,Charlottenburg-Wilmersdorf,0401,Charlottenburg,52.514088,13.304205,0101000020E6100000E59590B4C09B2A40382163A4CD41...,"{'city': 'Berlin', 'type': 'Wohnung', 'floor':...","{'bank': {'id': 'ban-416660137', 'name': 'targ..."
8,lis-WOH_1760_61_10777,https://www.immowelt.de/expose/529f581d-b85c-4...,listings,11004004,Charlottenburg-Wilmersdorf,0402,Wilmersdorf,52.497551,13.332489,0101000020E61000008E8DE5023CAA2A400B4DC8BDAF3F...,"{'city': 'Berlin', 'type': 'Wohnung', 'floor':...","{'bank': {'id': 'ban-35983986', 'name': 'inves..."
9,lis-STU_2580_85_10713,https://www.immowelt.de/expose/836ad724-e86d-4...,listings,11004004,Charlottenburg-Wilmersdorf,0402,Wilmersdorf,52.485510,13.318918,0101000020E6100000CF37FD3449A32A4017EF6C34253E...,"{'city': 'Berlin', 'type': 'Studio', 'floor': ...","{'bank': {'id': 'ban-28968292', 'name': 'berli..."


In [90]:
query = f"""
SELECT nearest_pois
FROM point_of_interest 
WHERE layer = 'listings';
"""

# Execute the query
with engine.connect() as conn:
    df= pd.read_sql(text(query), conn)
    conn.commit()  # commit the transaction
df
print(json.dumps(df.loc[5, "nearest_pois"], indent=2))

{
  "bank": {
    "id": "ban-1915389761",
    "name": "sparkasse",
    "address": {
      "street": "otto-suhr-allee",
      "house_number": null
    },
    "distance": 1011.56046421
  },
  "museum": {
    "id": "mus-9979815162",
    "name": "schloss charlottenburg",
    "address": {
      "street": "spandauer damm",
      "house_number": "10"
    },
    "distance": 485.49558849
  },
  "gallery": {
    "id": "gal-5265842119",
    "name": "galerie theis",
    "address": {
      "street": "schustehrus stra\u00dfe",
      "house_number": null
    },
    "distance": 840.65577041
  }
}


🧩 What’s Happening
For each listing row (WHERE l.category = 'listing'), Postgres runs three subqueries.

Each subquery filters the unified table by category (museum, gallery, bank).

The <-> operator finds the nearest neighbor using the GiST index on geom.

The results are bundled into a JSONB object and stored in nearest_pois.